# Workflow

## 1. Dataset Setup for Modeling

* Data Overview
* Feature and Target Definition
* Sample Construction
* Temporal Data Split
* Data Preprocessing

## 2. Model Development and Tuning

* Clustering: K-means
* Validation Portfolio Rule
* Pooled Elastic Net Tuning
* Clustered Elastic Net Tuning (Fixed K)

## 3. Final Model Estimation

* Historical-average benchmark
* OLS Baseline
* Pooled Elastic Net
* Clustered Elastic Net

## 4. Model Evaluation and Comparison

* 6-month Updating
* Test Evaluation
* Cluster Interpretation and Diagnostics

## 5. Portfolio Construction

* Final Portfolio Application
* Final Portfolio Performance
* Implementation Discussion

# 1. Dataset Setup for Modeling

This section prepares the stock-month panel used throughout the project. It covers the initial dataset inspection, definition of the target and feature sets, sample cleaning, data-type alignment, temporal splitting, and preprocessing setup.

The prediction target is the forward 1-month excess return, `ret_exc_lead1m`, which is also the required target in the project brief. The JKP documentation states that `eom` represents the information available by the end of the month, and that `ret_exc_lead1m` can be predicted directly using month-\(t\) characteristics without manually lagging them again.

At this stage, we construct the data objects that will feed into:
- model development,
- expanding-window hyperparameter tuning on the validation period,
- and final out-of-sample forecasting on the test period.

## 1. Data Overview

This subsection loads the cleaned US stock dataset and checks its basic structure, including:
- number of observations,
- number of variables,
- time coverage,
- and number of unique stocks.


### Import Module

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, r2_score
)
from sklearn.inspection import permutation_importance

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

### Load Dataset

In [ ]:
df = pd.read_parquet('jkp_data_us.parquet')

print(df.head(10))

In [ ]:
#print(df.sort_values(by='eom',ascending=False).head(5))
print(df.sort_values(by='eom',ascending=False).tail(5))


### Check Shape of Dataset

Before selecting features or splitting the sample, we inspect the overall structure of the dataset:
- available columns,
- total number of observations,
- total number of variables,
- and the number of unique stocks.

This gives a quick overview of the sample size and confirms that the dataset is organized as a stock-month panel suitable for cross-sectional return prediction.

In [ ]:
dfColumns = list(df.columns)
print(dfColumns)

In [ ]:
print("Shape of Dataset:", df.shape)
print("")
print("Observations No.:", df.shape[0])
print("")
print("Features No.:", df.shape[1])

In [ ]:
id = df["id"].unique()
print("No. of Stock:", len(id))

## 2. Feature and Target Definition

This subsection defines the target variable and the two feature groups used in the project.

The target variable is:
- `ret_exc_lead1m` :  the forward 1-month excess return.

We then separate the explanatory variables into:
- **prediction variables** :
    -  used in the pooled OLS, pooled Elastic Net, and clustered Elastic Net return-prediction models;
    - The prediction-variable set follows our theory-motivated design and covers momentum, liquidity/frictions, risk/volatility, profitability/quality, value/investment, and distress/issuance characteristics.

- **clustering variables** : 
    - used only for firm grouping in k-means;
    - The clustering-variable set is chosen to capture lifecycle, profitability–investment structure, financial fragility, and information environment, so that the resulting clusters can be interpreted economically.


In [ ]:
# Candidate prediction variables 

prediction_features = [
    'ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', #Momentum
    'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', #Liquidity/Frictions
    'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', #Risk/Volatilitye
    'gp_at', 'ocf_at', 'qmj', 'f_score', #Profitability/Quality
    'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'me', #Value/Investment
    'netdebt_me', 'o_score', 'z_score', 'eqnetis_at' #Distress/Issuance
]

target = 'ret_exc_lead1m'

In [ ]:
# Cluster variables are used only for k-means firm-type assignment

cluster_features = [
    'age', #Lifecycle
    'gp_at', 'ocf_at', 'sale_gr1', 'inv_gr1', 'dsale_dinv', #Profitability–Investment structure
    'netdebt_me', 'kz_index', 'z_score', #Financial fragility
    'niq_su', 'ni_inc8q' #Information environment
]

In [ ]:
# Union of prediction and clustering variables;
# We preprocess all variables jointly to --> ensure consistency across clustering and prediction tasks.
# This avoids discrepancies where the same economic variable is transformed differently across model components.

# dict.fromkeys(...) ： List value to dictionary key; as in dic no duplicates is allowed
#  - duplicates removed while preserving order
# --> {'ret_12_0': None,'gp_at': None...}  
# change dic back to list
features = list(dict.fromkeys(prediction_features + cluster_features))

In [ ]:
# Positive, highly skewed variables chosen for log1p transform

log_cols = ["me", "age"]

# Check the number of zero values before applying log1p
# This ensures the transformation is safe and helps identify potential data issues
for c in log_cols:
    print("No. of Zero of ",c,":",(df[c]==0).sum())

## 3. Sample Construction

This subsection constructs **sample data for modeling** (from the raw cleaned dataset).

We:
- drop observations with missing target values,
- keep only the identifier, date, target, and selected feature columns,
- convert `eom` to a proper datetime variable,
- sort the panel by stock and month,
- and coerce any remaining selected feature columns into numeric form where necessary.


### Drop Empty Target

Because the project is a supervised return-prediction exercise

--> Remove rows with missing values in:
- stock identifier,
- month-end date,
- or the target variable `ret_exc_lead1m`.

--> Inspect the remaining sample 

--> heck the time coverage of the usable stock-month panel

In [ ]:
# 1. check missing percentage of target
# df[target].isnull().mean()*100

# 2. remove rows missing id/eom/target
df_clear = df.dropna(subset=['id', 'eom', target])

# 3. confirm key variables no longer have missing values
df_clear[['id', 'eom', target]].isnull().mean()*100

In [ ]:
# Check % of missing values for each feature
(df_clear[features].isnull().mean()*100).sort_values()

In [ ]:
print(f"Data Time Range: {df_clear['eom'].min().date()} to {df_clear['eom'].max().date()}")

### Change Data Type and Drop Unselected Features

In [ ]:
df_clear['eom'] = pd.to_datetime(df_clear['eom'])
df_clear = df_clear.sort_values(['id', 'eom']).reset_index(drop=True)

In [ ]:
# The earliest observation of the stock with the smallest id
print(df_clear['eom'][0])

In [ ]:
# Keep only the key columns needed for modeling
df_clear = df_clear[['id', 'eom', target] + features].copy()
print(df_clear.head(5))

# missing_features = []
# for c in prediction_features+cluster_features:
#     if c not in df_clear.columns:
#         missing_features.append(c)
# print(missing_features)

In [ ]:
# Check data types of all columns.
print(df_clear.dtypes.sort_values())

# Identify selected feature columns that are stored as object type.
#   as object indicate numeric-looking variables may be stored as strings, not pure numeric
object_f = []
for f in features:
    if df_clear[f].dtypes == 'object':
        object_f.append(f)
print(object_f)
print(df_clear[object_f].head(5))

# Convert all selected features to numeric type.
# Any non-convertible values will be replaced with NaN.
for f in features:
    # errors='coerce': if a value cannot be converted into a number, pandas will turn it into NaN
    df_clear[f] = pd.to_numeric(df_clear[f], errors='coerce')
print(df_clear.dtypes.sort_values())
print(df_clear[object_f].head(5))


## 4. Temporal Data Split

This subsection creates the temporal data objects used in the rest of the notebook.

We follow the official split required by the project brief:
- Training: 2005-01 to 2015-12
- Validation: 2016-01 to 2018-12
- Test: 2019-01 to 2024-12


In [ ]:
# def split_train_valid_test(df: pd.DataFrame, validation_cutoff, test_cutoff):
#     """
#     Official project split:
#     Train: 2005-01 to 2015-12
#     Valid: 2016-01 to 2018-12
#     Test:  2019-01 to 2024-12
#     """
#     train_mask = (df['eom'] < validation_cutoff)
#     valid_mask = (df['eom'] >= validation_cutoff) & (df['eom'] < test_cutoff)
#     test_mask  = (df['eom'] >= test_cutoff)

#     train_df = df.loc[train_mask].copy()
#     valid_df = df.loc[valid_mask].copy()
#     test_df  = df.loc[test_mask].copy()

#     return train_df, valid_df, test_df


def split_expanding_block(df: pd.DataFrame, cutoff, end):
    """
    Split the dataset into:
    1. estimation window: all observations before the cutoff date
    2. prediction block: observations from cutoff date to before end date

    Example:
    cutoff = 2016-01-01
    end    = 2016-07-01

    estimation window: eom < 2016-01-01
    prediction block: 2016-01-01 <= eom < 2016-07-01
    """

    cutoff = pd.to_datetime(cutoff)
    end = pd.to_datetime(end)

    # Historical estimation window only
    train_mask = df['eom'] < cutoff

    # Next 6-month prediction block
    pred_mask = (df['eom'] >= cutoff) & (df['eom'] < end)

    train_df = df.loc[train_mask].copy()
    pred_df = df.loc[pred_mask].copy()

    return train_df, pred_df

In [ ]:
def date_updater(date, frequency=6, unit='months'):

    """
    unit: days, weeks, months, years
    
    e.g. 1 months, 6 months
    
    Move a date forward by the selected frequency.
    Used to define the next semiannual prediction block.
    """

    date = pd.to_datetime(date)

    if unit == 'days':
        return date + pd.Timedelta(days=frequency)
    elif unit == 'weeks':
        return date + pd.Timedelta(weeks=frequency)
    elif unit == 'months':
        return date + pd.DateOffset(months=frequency)
    elif unit == 'years':
        return date + pd.DateOffset(years=frequency)
    else:
        raise ValueError("unit must be one of: 'days', 'weeks', 'months', 'years'")


def expanding_window(df, start_date, end_date, frequency=6, unit='months'):


    # start_date： 2016-01-01 (val) & 2019-01-01(test)
    # end_date: 2019-01-01 (val) & 2025-01-01 (test)
    """
    Build expanding-window refit blocks.

    Each dictionary entry is:
    refit_date -> (estimation_window, next_prediction_block)

    At each refit date:
    - estimation_window uses all data before the refit date
    - prediction_block uses the next 6 months
    """

    # store all the expanding-window blocks
    # { Timestamp('2016-01-01'): (train_df_1, pred_df_1),
    #   Timestamp('2016-07-01'): (train_df_2, pred_df_2), ....}
    # Each key is a refit date
    update_set = {}

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    while start_date < end_date:
        next_date = date_updater(start_date, frequency, unit)

        # split to 2 set : train_df, pred_df
        update_set[start_date] = split_expanding_block(
            df=df,
            cutoff=start_date,
            end=next_date
        )

        start_date = next_date

    return update_set

In [ ]:
# Creates validation-period blocks
#   Each block is 6 months
valid_update_set = expanding_window(
    df_clear,
    start_date=pd.to_datetime('2016-01-01'),
    end_date=pd.to_datetime('2019-01-01'),
    frequency=6,
    unit='months'
)

In [ ]:
# Creates test-period blocks
#    Each block is 6 months
test_update_set = expanding_window(
    df_clear,
    start_date=pd.to_datetime('2019-01-01'),
    end_date=pd.to_datetime('2025-01-01'),
    frequency=6,
    unit='months'
)

In [ ]:
# Summary of Splitting 
def summarize_split(name, df):
    print(f"{name}: {df['eom'].min().date()} to {df['eom'].max().date()}, rows={len(df):,}, ids={df['id'].nunique():,}")


for k, (train, valid) in valid_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Valid {k}", valid)
    print("")

for k, (train, test) in test_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Test  {k}", test)
    print("")


# Expanding Window Includes More Stocks (# ids increase) Over Time
#   these are stocks that didn’t exist earlier or were not included yet.

# Stocks (# ids smaller in prediction set)May Have Missing Observations
#   Some IPOs appear after y.
#   Some stocks may be delisted or missing in early years.


## 5. Data Preprocessing

This subsection defines the preprocessing pipeline applied to the selected modeling variables.

Following the project brief, preprocessing includes:
- log transformation of selected skewed variables  
    - Avoid Large outliers dominate coefficient estimation -> stable coeffienct
- winsorization of outliers
    - more robust to extreme outliers, avoid dominate
- imputation of missing values
    - Median is not senstive to oulier (vs mean), ensures all rows are usable
- standardization of predictors
    - fair contribution of all features, avoid dominate
    - both Kmean (distance) & regression sensitive to scale

All preprocessing objects are always fit on the relevant estimation window only.


In [ ]:
# Custom preprocessing pipeline:
# fit all preprocessing objects on the training segment only, then apply them to later blocks.
# This avoids look-ahead bias in winsorization, imputation, and standardization.

# Why class:
# - For every expanding-window refit date, we need to do the same thing:
# SP = StockPreprocessor(...)
# SP.fit(train_df)
# train_processed = SP.transform(train_df)
# pred_processed = SP.transform(pred_df)
class StockPreprocessor:
    """
    Steps:
    1. apply log(1+x) transformation to selected skewed variables
    2. winsorize features using training-set quantile bounds
    3. impute missing values using training-set medians
    4. standardize using training-set mean and standard deviation
    """


    #   take Arguments for object
    #   so every method in this class can access them 
    def __init__(
        self,
        feature_cols,
        target_col,
        log1p_cols=None,
        id_col = 'id',
        date_col = 'eom',
        winsor_lower=0.01,
        winsor_upper=0.99
    ):
        
        # attributes
        self.id_col = id_col
        self.date_col = date_col
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.log1p_cols = log1p_cols if log1p_cols is not None else []
        self.winsor_lower = winsor_lower
        self.winsor_upper = winsor_upper

        # These objects are learned from training data only
        self.X_lower_bounds_ = None
        self.X_upper_bounds_ = None
        self.y_lower_bounds_ = None
        self.y_upper_bounds_ = None
        self.imputer_ = None
        self.scaler_ = None

# Method(function)
    def _apply_log1p(self, df):
        
        # Apply log1p only to selected positive, highly skewed variables.
        # Their transformed names become log_<variable> in the processed output.

        out = df.copy()

        for col in self.log1p_cols:
            if col in out.columns:
                out[col] = np.log1p(out[col])
        
        return out

    def _winsorize_with_fitted_bounds(self, df):
        
        # Winsorization bounds are estimated from the training segment only
        # and then applied transformation to the corresponding validation / test block.

        X_wins = np.clip(df[self.feature_cols].values.astype(float), self.X_lower_bounds_, self.X_upper_bounds_)

        y_wins = df[self.target_col] #.clip(self.y_lower_bounds_, self.y_upper_bounds_)

        return X_wins, y_wins

    def fit(self, train_df):
        """
        Fit preprocessing objects using training data only.
        """
        X = train_df[self.feature_cols].copy()
        y = train_df[self.target_col].copy()

        X = self._apply_log1p(X)

        # Step 1: compute winsorization bounds from training only
        self.X_lower_bounds_ = X.quantile(self.winsor_lower).values.astype(float)
        self.X_upper_bounds_ = X.quantile(self.winsor_upper).values.astype(float)

        # self.y_lower_bounds_ = float(y.quantile(self.winsor_lower))
        # self.y_upper_bounds_ = float(y.quantile(self.winsor_upper))
        
        # Combine target + features
        df = pd.concat([y, X], axis=1)

        # Apply winsorization to the training data
        X, y = self._winsorize_with_fitted_bounds(df)

        # Step 2: fit median imputer on training only
        #              Replace missing values in training set
        self.imputer_ = SimpleImputer(strategy="median")

        X_imputed = pd.DataFrame(self.imputer_.fit_transform(X), columns=self.feature_cols, index=train_df.index)

        # Step 3: fit scaler on training only
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X_imputed)

        return self

    def transform(self, df):
        """
        Apply fitted preprocessing objects to any new dataset
        such as validation or test.
        """
        X = df[self.feature_cols].copy()
        y = df[self.target_col].copy()

        X = self._apply_log1p(X)
        
        out = pd.concat([y, X], axis=1)

        # Apply the same winsorization bounds learned from training
        X, y = self._winsorize_with_fitted_bounds(out)

        # Apply the same training-fitted median imputer
        X_imputed = pd.DataFrame(self.imputer_.transform(X), columns=self.feature_cols, index=df.index)

        # Apply the same training-fitted scaler
        X_scaled = self.scaler_.transform(X_imputed)

        # Put transformed features back into a DataFrame
        X_out = pd.DataFrame(X_scaled, columns=self.feature_cols, index=out.index)

        # Return original dataset with transformed feature columns replaced
        out = pd.concat([df[[self.id_col, self.date_col]].copy(), y, X_out], axis=1)

        # After preprocessing, 'me' and 'age' become 'log_me' and 'log_age'
        for x in self.log1p_cols:
            if x in out.columns:
                out = out.rename(columns={x: f"log_{x}"})
        return out

    def fit_transform(self, train_df):
        """
        Convenience method: fit on training data and transform training data.
        """
        self.fit(train_df)
        return self.transform(train_df)

### Preprocessing Data

In [ ]:
# All downstream model code should use model_features, not raw features

model_features = features.copy()
for i in log_cols:
    model_features[model_features.index(i)] = f"log_{i}"
print(model_features)

In [ ]:
log_prediction_features = prediction_features.copy()
for i in log_cols:
    if i in log_prediction_features:
        log_prediction_features[log_prediction_features.index(i)] = f"log_{i}"

log_cluster_features = cluster_features.copy()
for i in log_cols:
    if i in log_cluster_features:
        log_cluster_features[log_cluster_features.index(i)] = f"log_{i}"

print(log_prediction_features)
print(log_cluster_features)

In [ ]:
# At each refit date, we re-estimate preprocessing objects, centroids, and coefficients
# using only data available up to that date, then forecast the next 6 months.

# Dictionary of 6-month expanding-window refit blocks for the validation stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

valid_update_window = {}
for start_date in valid_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(valid_update_set[start_date][0])
    valid_update_window[start_date] = [SP.transform(valid_update_set[start_date][0]),SP.transform(valid_update_set[start_date][1])]

In [ ]:
# Dictionary of 6-month expanding-window refit blocks for the test stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

test_update_window = {}
for start_date in test_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(test_update_set[start_date][0])
    test_update_window[start_date] = [SP.transform(test_update_set[start_date][0]),SP.transform(test_update_set[start_date][1])]

### Check Preprocessed Data

hhhhh


In [ ]:
mean_pre_valid = 0
for x in valid_update_window.keys():
    mean_pre_valid += valid_update_window[x][0][model_features].mean()

print("Train-Valid Set Mean:\n", mean_pre_valid/len(valid_update_window), sep='')

In [ ]:
std_pre_valid = 0
for x in valid_update_window.keys():
    std_pre_valid += valid_update_window[x][0][model_features].std(ddof=0)

print("Train-Valid Set Std:\n", std_pre_valid/len(valid_update_window), sep='')

In [ ]:
for x in valid_update_window.keys():
    print('Shape of train-valid set', x,':', valid_update_window[x][0].shape,'/', valid_update_window[x][1].shape)

In [ ]:
mean_pre_test = 0
for x in test_update_window.keys():
    mean_pre_test += test_update_window[x][0][model_features].mean()

print("Train-Test Set Mean:\n", mean_pre_test/len(test_update_window), sep='')

In [ ]:
std_pre_test = 0
for x in test_update_window.keys():
    std_pre_test += test_update_window[x][0][model_features].std(ddof=0)

print("Train-Test Set Std:\n", std_pre_test/len(test_update_window), sep='')

In [ ]:
for x in test_update_window.keys():
    print('Shape of train-test set', x,':', test_update_window[x][0].shape,'/', test_update_window[x][1].shape)

# 2. Model Development and Tuning

This section finalizes the model specification using the processed data, the 2005–2015 training sample, and the official 2016–2018 validation period. Since preprocessing and temporal splitting have already been completed, we begin directly with model-development choices.

Our goal is to compare a non-clustered regularized benchmark against a cluster-conditional regularized model. The pooled Elastic Net and clustered Elastic Net use different design matrices, so they are tuned separately. The pooled Elastic Net uses only base predictors, while the clustered Elastic Net uses base predictors, cluster dummies, and predictor-by-cluster interactions.

We first choose the number of clusters \(K\) using the training sample only, based on clustering diagnostics and economic interpretability. We then use the validation-period expanding-window forecasting exercise to tune the regularization hyperparameters of the pooled and clustered Elastic Net models. Hyperparameters are selected using validation portfolio Sharpe ratio as the main criterion.

At the end of this section, we fix:
- the chosen number of clusters \(K^*\),
- the tuned pooled Elastic Net hyperparameters,
- the tuned clustered Elastic Net hyperparameters.

These hyperparameters remain fixed afterward. In later forecasting stages, we update preprocessing objects, cluster centroids, and model coefficients every 6 months.

## 2.1. Clustering: K-means

### Objective
Group firms into economically meaningful clusters before estimating the cluster-conditional prediction model.

### Clustering variables
We use a smaller firm-type variable set designed to capture:
- lifecycle,
- profitability-investment structure,
- financial fragility,
- information environment.

Confirmed clustering variables:
- `age`
- `gp_at`, `ocf_at`, `sale_gr1`, `inv_gr1`, `dsale_dinv`
- `netdebt_me`, `kz_index`, `z_score`
- `niq_su`, `ni_inc8q`

### Method
We use k-means clustering with candidate cluster counts:
- \(K = 2, 3, 4, 5, ...\)

### What this section should do
- construct the clustering feature matrix from the processed clustering variables,
- fit k-means on the 2005–2015 training sample for each candidate `K`,
- compare candidate values using:
  - elbow / within-cluster variation diagnostics,
  - cluster-size balance,
  - centroid distinctness and economic interpretability,
- choose one final \(K^*\),
- define the procedure for assigning later observations to the nearest centroids.

### Why we do it
Our project hypothesis is that firms may differ systematically in the way characteristics map into future returns. Clustering provides a data-driven but interpretable firm-type classification that can be used to allow slope heterogeneity in the final prediction model.

### Important note
We do **not** use PCA to construct the clusters. K-means is run directly on the standardized clustering variables so that the cluster inputs remain economically interpretable.

### Useful visualizations for cluster construction
- elbow plot across candidate \(K\),
- cluster-size bar chart for each candidate \(K\),
- centroid summary table or heatmap for each candidate \(K\).

## 2.2. Validation Portfolio Rule

### Objective
Define the portfolio-construction rule used during hyperparameter tuning.

### Portfolio rule
To tune the regularized models using validation Sharpe ratio, we must first fix a common mapping from stock-level predictions to portfolio weights. Following the logic in the skeleton code, we use a rank-based zero-cost long–short rule.

For each month in the validation period:
1. rank stocks by predicted return,
2. center the ranks so that the portfolio is dollar-neutral,
3. normalize the weights,
4. optionally cap the maximum absolute weight per stock,
5. compute the monthly long–short return using the realized next-month excess return.

### Why we use this rule
This approach is robust to extreme prediction outliers because it depends on cross-sectional ranking rather than raw predicted-return magnitudes. It also keeps the tuning exercise comparable across hyperparameter settings and across models.

### What this section should do
- define the portfolio-weighting function used in validation tuning,
- apply the same rule for pooled Elastic Net and clustered Elastic Net,
- compute monthly validation portfolio returns from model predictions,
- use the resulting annualized Sharpe ratio as the main hyperparameter-tuning criterion.

### Important note
This section only defines the portfolio rule used for validation-time model selection. The full portfolio evaluation and reporting for the final models is presented later in the test-period portfolio section.

### Useful visualizations
- schematic illustration of prediction → ranking → weights,
- histogram of portfolio weights in a sample month,
- monthly number of stocks in the long and short sides.

## 2.3. Pooled Elastic Net Tuning

### Objective
Tune the regularized non-clustered benchmark model.

### Model idea
The pooled Elastic Net uses the selected prediction variables only, with no cluster identifiers and no interaction terms. It serves as the regularized benchmark against which we later compare the clustered Elastic Net.

### Inputs
The pooled predictor set is the theory-motivated candidate set covering:
- momentum,
- liquidity/frictions,
- risk/volatility,
- profitability/quality,
- value/investment,
- distress/issuance.

### Tuning task
Using the official validation period in an expanding-window forecasting exercise, we tune:
- `alpha_pooled`
- `l1_ratio_pooled`

### What this section should do
- construct the pooled Elastic Net design matrix,
- define the tuning grid for `alpha` and `l1_ratio_pooled`,
- for each candidate pair, run the full validation-period expanding-window forecasting loop,
- construct validation-period portfolios from the resulting predictions,
- select the pooled Elastic Net hyperparameters using validation portfolio Sharpe ratio.

### Why we do it
This model provides a fair regularized benchmark. It allows us to distinguish:
- the effect of regularization alone,
from
- the additional effect of introducing cluster-conditioned predictor slopes.

### Useful visualizations
- validation-Sharpe heatmap across `alpha` and `l1_ratio_pooled`,
- coefficient-path plot against `l1_ratio_pooled`,
- table of active predictors under the selected pooled Elastic Net.

## 2.4. Clustered Elastic Net Tuning (Fixed K)

### Objective
Tune the main model of the project: the cluster-conditional Elastic Net.

### Model idea
The clustered Elastic Net uses:
- base prediction variables,
- cluster dummies,
- predictor-by-cluster interactions.

This allows the predictive effect of each characteristic to vary across firm clusters.

### Tuning task
With \(K^*\) fixed from the clustering step, we tune:
- `alpha_clustered`
- `l1_ratio_clustered`

using the validation-period expanding-window forecasting exercise.

### What this section should do
For each candidate combination of `alpha` and `l1_ratio_clustered`:
1. at each validation refit date, fit k-means on the current expanding estimation window using fixed \(K^*\),
2. assign validation-block observations to the nearest centroids,
3. construct the final clustered design matrix,
4. fit clustered Elastic Net on the current estimation window,
5. generate predictions for the next validation block,
6. construct the validation-period portfolio,
7. evaluate overall validation Sharpe ratio across all validation blocks.

At the end of this section, we select:
- `alpha_clustered*`,
- `l1_ratio_clustered*`.

### Why we do it
Once cluster dummies and predictor-by-cluster interactions are added, the number of parameters grows quickly. Elastic Net is used here to shrink the expanded interaction design and reduce the effective number of active parameters while preserving flexibility in the final forecasting model.

### Useful visualizations
- validation-Sharpe heatmap for clustered Elastic Net,
- number of active coefficients under the chosen clustered model,
- heatmap of top interaction coefficients.

# 3. Final Model Estimation

This section estimates the finalized comparison models using the tuned specifications from the previous section. No further hyperparameter search is conducted here.

The final comparison set includes:
- Historical-average benchmark,
- Pooled OLS,
- Pooled Elastic Net,
- Clustered Elastic Net.

All models will later be run under the same 6-month expanding-window update protocol.

## 3.1. Historical-average benchmark

### Model idea
At each refit date, predict every stock-month observation using the average `ret_exc_lead1m` in the current estimation window.

This historical-average benchmark does not use stock characteristics and does not generate cross-sectional variation in predictions. Its main role is to provide the null benchmark for out-of-sample \(R^2\).

### Useful visualizations
- cumulative return curve for the benchmark portfolio,
- histogram of benchmark predictions,
- summary statistics of stock-level historical means.

## 3.2. OLS Baseline

### Objective
Estimate the pooled linear benchmark without regularization or cluster structure.

### Model idea
Use the processed predictor set in a pooled cross-sectional OLS regression.

### What this section should do
- build the pooled OLS design matrix,
- fit OLS on the current estimation window,
- generate stock-level predictions for the next forecast block.

### Why we do it
This is the required linear baseline and provides the benchmark for judging whether regularization and cluster-conditioning add value.

### Useful visualizations
- coefficient table for a selected refit date,
- coefficient stability plot across refit windows,
- predicted-vs-realized scatterplot for a validation block.

## 3.3. Pooled Elastic Net

### Objective
Estimate the tuned regularized non-clustered benchmark.

### Model idea
Use the same base predictors as the clustered model, but without cluster dummies or interaction terms.

### Hyperparameters
Use the tuned:
- `alpha_pooled*`
- `l1_ratio_pooled*`

### What this section should do
- fit pooled Elastic Net on each estimation window using the fixed pooled hyperparameters,
- generate stock-level predictions for the next forecast block.

### Why we do it
This model isolates the effect of regularization. Comparing it with clustered Elastic Net tells us whether cluster-conditioning adds value beyond a non-clustered regularized benchmark.

### Useful visualizations
- active-coefficient count over time,
- comparison of active predictors vs OLS coefficients,
- coefficient-path plot for the selected pooled model.

## 3.4. Clustered Elastic Net

### Objective
Estimate the main model of the project using the tuned clustered specification.

### Model idea
The final clustered Elastic Net includes:
- base predictors,
- cluster dummies,
- predictor-by-cluster interactions.

### Hyperparameters
Use the tuned:
- `K^*`
- `alpha_clustered*`
- `l1_ratio_clustered*`

### What this section should do
- re-estimate k-means centroids with fixed \(K^*\),
- assign cluster labels,
- construct cluster dummies and predictor-by-cluster interactions,
- fit clustered Elastic Net with the fixed clustered hyperparameters,
- generate stock-level predictions for the next forecast block.

### Why we do it
This is the core model used to test whether predictor effects differ across firm types and whether allowing such heterogeneity improves prediction and portfolio performance.

### Useful visualizations
- active interaction count over time,
- top interaction coefficients table,
- coefficient heatmap by predictor and cluster,
- comparison of predicted-return distributions from pooled vs clustered Elastic Net.

# 4. Model Evaluation and Comparison

This section evaluates the finalized models under the common forecasting protocol and reports the final out-of-sample predictive results. Since the validation period is now used for model tuning, the main formal evaluation is conducted on the untouched test period. We also summarize how the 6-month expanding-window updating scheme is implemented and interpret the economic meaning of the final clustered model.

## 4.1. 6-month Updating

### Objective
Implement the agreed forecasting protocol used in both the validation tuning exercise and the final test-period evaluation.

### Update rule
- expanding window,
- refit every 6 months.

### What updates every 6 months
- preprocessing objects,
- cluster centroids,
- model coefficients.

### What remains fixed
- chosen number of clusters \(K^*\),
- pooled Elastic Net hyperparameters,
- clustered Elastic Net hyperparameters,
- predictor sets,
- overall model structure.

### Why we do it
This allows the models to adapt to newly available data while preserving a fixed design and avoiding repeated redefinition of the full methodology. The same update logic is used consistently throughout the forecasting exercise, with only past information available at each refit date.

## 4.2. Test Evaluation

### Objective
Evaluate final out-of-sample predictive performance on the official test period.

### Test period
- 2019-01 to 2024-12

### What this section should do
- run the finalized models using the same 6-month expanding-window protocol,
- generate monthly stock-level predictions for each model,
- compute the final out-of-sample predictive metrics on the untouched test sample.

### Models evaluated
- Historical-average benchmark
- Pooled OLS
- Pooled Elastic Net
- Clustered Elastic Net

### Required predictive output
Report test out-of-sample \(R^2\) for each model.

### Why this matters
This is the main formal performance evaluation stage of the project, because the validation period has already been used for model tuning.

### Useful visualizations
- test \(R^2\) comparison chart,
- predicted-return decile vs realized-return plot,
- predicted vs realized return scatter or binned plot.

## 4.3. Cluster Interpretation and Diagnostics

### Objective
Interpret the selected clustering structure and explain the economic meaning of the final clustered model.

### What this section should do
- summarize the selected centroids using the clustering variables,
- label each cluster economically,
- compare cluster sizes under the selected \(K^*\),
- examine which predictor-by-cluster effects remain active in the final clustered Elastic Net,
- assess whether the final model tells a coherent finance story.

### Why this matters
The clustered model is valuable only if the selected clusters are not just statistically distinct, but also economically meaningful. This section connects the final ML results back to the underlying finance question.

### Useful visualizations
- centroid heatmap,
- cluster-size distribution,
- boxplots of important clustering variables by cluster,
- PCA scatter plot colored by cluster for visualization only,
- heatmap of active predictor-by-cluster coefficients,
- ranked table of the most important interaction effects.

# 5. Portfolio Construction

This section applies the previously defined portfolio rule to the final test-period predictions from each model. The purpose here is not to redefine the portfolio formation method, but to evaluate the economic performance of the benchmark and machine-learning models under a common trading rule.

## 5.1 Final Portfolio Application

### Objective
Apply the portfolio rule defined earlier in the tuning stage to the final test-period predictions.

### What this section should do
- take the monthly stock-level predictions from each finalized model,
- apply the common rank-based long–short weighting rule,
- compute the resulting monthly portfolio returns over the test period.

### Why this section is separate
The portfolio rule itself is defined earlier because it is already needed for Sharpe-based hyperparameter tuning. Here, we simply apply that same fixed rule to the final out-of-sample predictions so that portfolio performance is evaluated consistently across models.

## 5.2. Final Portfolio Performance

### Objective
Evaluate the economic performance of the portfolios generated by each final prediction model.

### What this section should do
- summarize monthly test-period long–short returns for each model,
- compare benchmark, linear, and machine-learning strategies on an economic basis.

### Report
For each model, report:
- annualized mean excess return,
- annualized volatility,
- Sharpe ratio.

### Useful visualizations
- cumulative return curves across models,
- bar chart of annualized return / volatility / Sharpe,
- drawdown plot,
- monthly return distribution plot.

## 5.3. Implementation Discussion

### Objective
Discuss whether the test-period portfolio results are likely to survive real-world trading frictions.

### What this section should do
Provide a qualitative discussion of:
- turnover,
- transaction costs,
- liquidity constraints,
- concentration risk,
- whether the strategy remains economically plausible after implementation frictions.

### Why we do it
The project guideline explicitly asks for a qualitative discussion of implementation issues, even if they are not modeled quantitatively.